# 01 — Exploratory Data Analysis
### Credit Card Fraud Detection

Goal of this notebook: understand the shape, quality, and class-separability
of the data *before* making any feature-engineering or modeling decisions in
notebooks 02-04.

Because `src/` was installed in editable mode (`pip install -e .` in Phase 2),
imports below work regardless of this notebook's working directory — no
`sys.path` hacks needed. Just make sure the kernel is set to
**Python (fraud-detection)**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import TARGET_COL, FIGURES_DIR
from src.data import load_raw_data

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)

%matplotlib inline


## 1. Load Data

Requires `make download-data` to have been run already (Phase 3).

In [ ]:
df = load_raw_data()
print(f"Shape: {df.shape}")
df.head()


## 2. Structural Overview

In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
n_missing = df.isna().sum().sum()
print(f"Total missing values across the dataset: {n_missing}")


### Duplicate rows

A known quirk of this dataset — worth quantifying before deciding anything.

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes} ({n_dupes / len(df) * 100:.3f}% of dataset)")

dupes_by_class = df[df.duplicated(keep=False)].groupby(TARGET_COL).size()
print("\nDuplicated rows by class:")
print(dupes_by_class)


> **Note:** don't decide to drop duplicates yet — that decision (and the
> reasoning behind it) belongs in notebook 02. Dropping duplicated fraud rows
> in particular could remove real, if repeated, signal rather than noise.

## 3. Class Imbalance

The central challenge of this project.

In [ ]:
class_counts = df[TARGET_COL].value_counts()
class_pct = df[TARGET_COL].value_counts(normalize=True) * 100

summary = pd.DataFrame({"count": class_counts, "percentage": class_pct.round(4)})
summary.index = summary.index.map({0: "Legitimate", 1: "Fraud"})
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x=TARGET_COL, ax=axes[0])
axes[0].set_title("Class Distribution (linear scale)")
axes[0].set_xticklabels(["Legitimate", "Fraud"])

sns.countplot(data=df, x=TARGET_COL, ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Class Distribution (log scale)")
axes[1].set_xticklabels(["Legitimate", "Fraud"])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_class_distribution.png", bbox_inches="tight")
plt.show()


**Why this matters:** on the linear-scale panel, the fraud bar is visually
almost nonexistent at ~0.17% of transactions. This is the whole reason
accuracy is the wrong metric for this project — a model that predicts
"legitimate" for every single transaction scores ~99.83% accuracy while
catching zero fraud. PR-AUC (Phase 7-8) is what we optimize instead.

## 4. Time Feature

`Time` is seconds elapsed since the first transaction in the dataset (spans ~48 hours).

In [ ]:
df["Hour"] = (df["Time"] % 86400) // 3600

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.loc[df[TARGET_COL] == 0, "Hour"].hist(bins=24, ax=axes[0], alpha=0.6, label="Legitimate", density=True)
df.loc[df[TARGET_COL] == 1, "Hour"].hist(bins=24, ax=axes[0], alpha=0.6, label="Fraud", density=True)
axes[0].set_title("Transaction density by hour-of-day")
axes[0].set_xlabel("Hour")
axes[0].legend()

sns.histplot(data=df, x="Time", hue=TARGET_COL, bins=48, ax=axes[1], stat="density", common_norm=False)
axes[1].set_title("Transaction density over full Time range")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_time_distribution.png", bbox_inches="tight")
plt.show()


**What to look for:** does fraud cluster at particular hours relative to
legitimate traffic? If the two curves in the left panel diverge meaningfully,
`Hour` is a genuinely useful engineered feature for notebook 02. If they
largely overlap, it may add little beyond what `Time` already gives the
model.

## 5. Amount Feature

In [ ]:
df.groupby(TARGET_COL)["Amount"].describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=df, x=TARGET_COL, y="Amount", ax=axes[0])
axes[0].set_title("Amount by class (raw)")
axes[0].set_xticklabels(["Legitimate", "Fraud"])

sns.boxplot(data=df, x=TARGET_COL, y="Amount", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Amount by class (log scale)")
axes[1].set_xticklabels(["Legitimate", "Fraud"])

df["Amount_log"] = np.log1p(df["Amount"])
sns.kdeplot(data=df, x="Amount_log", hue=TARGET_COL, ax=axes[2], common_norm=False)
axes[2].set_title("log1p(Amount) density by class")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_amount_distribution.png", bbox_inches="tight")
plt.show()

print(f"Amount skewness (raw):    {df['Amount'].skew():.2f}")
print(f"Amount skewness (log1p):  {df['Amount_log'].skew():.2f}")


**Why `log1p`:** raw `Amount` is heavily right-skewed — a small number of
large transactions dominate the tail and would otherwise dominate any
distance- or variance-based scaler. This is exactly why `src/features.py`
(Phase 4) derives `Amount_log` and scales that, rather than feeding raw
`Amount` straight into `RobustScaler`.

## 6. V1-V28 (PCA) Features

Since these are already PCA components, we can't attach real-world meaning
to any individual one — but the *shape* of separation between the fraud and
legitimate curves in each panel tells us how discriminative that component
is for this task.

In [ ]:
v_cols = [f"V{i}" for i in range(1, 29)]

fig, axes = plt.subplots(7, 4, figsize=(20, 28))
axes = axes.flatten()

for i, col in enumerate(v_cols):
    sns.kdeplot(
        data=df[df[TARGET_COL] == 0], x=col, ax=axes[i],
        label="Legitimate", fill=True, alpha=0.4, common_norm=False,
    )
    sns.kdeplot(
        data=df[df[TARGET_COL] == 1], x=col, ax=axes[i],
        label="Fraud", fill=True, alpha=0.4, common_norm=False,
    )
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_v_features_grid.png", bbox_inches="tight")
plt.show()


**How to read this grid:** panels where the fraud curve is clearly shifted
or narrower relative to the legitimate curve are doing a lot of work for the
model. Panels where the two curves nearly overlap are close to noise for
this specific task — the next section quantifies this instead of eyeballing
28 subplots.

## 7. Statistical Separability Ranking

Quantifying what the grid above shows visually, using two independent measures:
- **KS statistic**: max distance between the two classes' empirical CDFs (higher = more separable)
- **Point-biserial correlation**: correlation between the (binary) target and each continuous feature

In [ ]:
ks_results = []
for col in v_cols + ["Time", "Amount", "Amount_log"]:
    legit_vals = df.loc[df[TARGET_COL] == 0, col]
    fraud_vals = df.loc[df[TARGET_COL] == 1, col]
    ks_stat, ks_pval = stats.ks_2samp(legit_vals, fraud_vals)
    pb_corr, pb_pval = stats.pointbiserialr(df[TARGET_COL], df[col])
    ks_results.append({
        "feature": col,
        "ks_statistic": ks_stat,
        "ks_pvalue": ks_pval,
        "point_biserial_corr": pb_corr,
    })

ks_df = pd.DataFrame(ks_results).sort_values("ks_statistic", ascending=False).reset_index(drop=True)
ks_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(data=ks_df, y="feature", x="ks_statistic", ax=ax)
ax.set_title("Feature separability: KS statistic (fraud vs. legitimate)")
ax.set_xlabel("KS statistic (higher = more separable)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_feature_separability.png", bbox_inches="tight")
plt.show()

print("Top 10 most discriminative features by KS statistic:")
print(ks_df.head(10)[["feature", "ks_statistic", "point_biserial_corr"]].to_string(index=False))


**Why this matters for later phases:** keep this ranking in mind for
Phase 8 — SHAP feature importances should broadly agree with it. If SHAP
ranks a feature this analysis calls near-noise as highly important, that's
worth a second look (possible overfitting, or a real interaction effect
this univariate analysis can't see).

## 8. Correlation Structure

In [ ]:
corr = df[v_cols + ["Time", "Amount", TARGET_COL]].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, square=True, cbar_kws={"shrink": 0.7})
ax.set_title("Correlation matrix: V1-V28, Time, Amount, Class")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_correlation_heatmap.png", bbox_inches="tight")
plt.show()


**Expected pattern:** because V1-V28 come from PCA, they're orthogonal to
each other by construction — near-zero off-diagonal correlation among them
is expected and confirms the PCA did what it claims. The interesting row/
column to actually read is `Class`, which should echo the KS ranking above.

## 9. Outlier Diagnostic (informational only)

**Important framing:** in fraud detection, statistical "outliers" by the standard IQR rule often correlate with the exact fraud cases we're trying to detect — not noise to clean away. This cell is diagnostic, not a cleaning step. We will not be dropping IQR outliers in notebook 02.

In [ ]:
def iqr_outlier_pct(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).mean() * 100

outlier_summary = pd.DataFrame({
    "outlier_pct_legit": [iqr_outlier_pct(df.loc[df[TARGET_COL] == 0, c]) for c in v_cols],
    "outlier_pct_fraud": [iqr_outlier_pct(df.loc[df[TARGET_COL] == 1, c]) for c in v_cols],
}, index=v_cols).sort_values("outlier_pct_fraud", ascending=False)

outlier_summary.head(10)


**What to look for:** if `outlier_pct_fraud` is dramatically higher than
`outlier_pct_legit` for the top-ranked features here, and those same
features also scored high on the KS ranking in section 7, that's
confirmation the signal driving separability *is* what IQR flags as
"extreme" — reinforcing that outlier removal would be actively harmful
here, not helpful.

## 10. Key Findings

1. **Class imbalance:** Highly imbalanced — 492 fraud cases out of 284,807 transactions (0.1727%). Primary optimization metric must be PR-AUC rather than ROC-AUC or Accuracy.
2. **Duplicates:** 1,081 duplicate rows exist (32 fraud, 1,822 legit). Duplicate fraud cases will be kept because identical transaction patterns can represent automated fraud attempts rather than redundant data error.
3. **Time:** Fraud shows elevated density during low-volume night hours (hours 2–3). `Hour` is a valid candidate for feature engineering.
4. **Amount:** Raw `Amount` has extreme positive skewness (16.98). Applying `log1p` reduces skewness to 0.16 and creates a balanced distribution suitable for model scaling.
5. **Most discriminative V-features:** `V14`, `V10`, `V12`, `V17`, `V11`, and `V4` show strong class separation under KS testing and KDE distribution plots.
6. **Least discriminative V-features:** `V13`, `V15`, `V22`, `V25`, and `V26` show near-identical density profiles across classes.
7. **Correlation structure:** `V1`–`V28` exhibit zero off-diagonal correlation, verifying orthogonal PCA components.
8. **Outlier diagnostic:** "Outliers" defined by IQR rules contain a significant portion of the true fraud signal. Statistical outlier removal will be avoided to prevent destroying positive labels.

### Decisions for Feature Engineering & Pipeline (Phase 6)
- [ ] Keep duplicate rows to preserve rare fraud signals.
- [ ] Use `Amount_log` instead of raw `Amount`.
- [ ] Retain `Hour` feature derived from `Time % 86400 // 3600`.
- [ ] Do NOT strip IQR outliers from training data.